In [5]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 7, Finished, Available, Finished, False)

In [6]:
df = spark.sql("SELECT * FROM myadls.swiggy_incremental")
display(df)

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 8, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 715899ea-eb78-40da-bf6c-ff57a1bbac88)

In [7]:
df.printSchema()

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 9, Finished, Available, Finished, False)

root
 |-- order_id: string (nullable = true)
 |-- order_time: timestamp (nullable = true)
 |-- delivery_time: timestamp (nullable = true)
 |-- restaurant: string (nullable = true)
 |-- cuisine: string (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- rating: string (nullable = true)
 |-- location: string (nullable = true)
 |-- city: string (nullable = true)
 |-- ordered_qty: integer (nullable = true)
 |-- book_table: string (nullable = true)
 |-- online_order: string (nullable = true)
 |-- distance_km: double (nullable = true)



# IDENTIFYING MISSING VALUES

In [8]:
for c in df.columns:
    nulls=df.filter(col(c).isNull()).count()
    print(c,nulls)

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 10, Finished, Available, Finished, False)

order_id 0
order_time 0
delivery_time 0
restaurant 0
cuisine 0
total_amount 0
rating 0
location 0
city 0
ordered_qty 0
book_table 0
online_order 0
distance_km 0


# HANDLING MISSING VALUES

In [9]:
df=df.fillna({
    "restaurant":"NA",
    "cuisine":"NA",
    "location":"NA",
    "city":"NA",
    "book_table":"NA",
    "online_order":"NA"
})

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 11, Finished, Available, Finished, False)

In [10]:
df=df.fillna({
    "ordered_qty":0,
    "distance_km":0,
    "total_amount":0
})

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 12, Finished, Available, Finished, False)

# STRING STANDARDIZATION

In [11]:
df = df.withColumn("city", trim(col("city")))
df = df.withColumn("city", initcap(col("city")))
df = df.withColumn("city", regexp_replace(col("city"), "Bnegaluru", "Bengaluru"))

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 13, Finished, Available, Finished, False)

# NUMERICAL TRANSFORMATION

In [12]:
df = df.withColumn("rating",when(col("rating") == "NEW", None).otherwise(col("rating")))
df=df.withColumn("rating_numeric",split(col("rating"),"/").getItem(0).cast("float"))
df=df.drop("rating")

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 14, Finished, Available, Finished, False)

In [13]:
df = df.withColumn("total_amount", abs(col("total_amount")))

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 15, Finished, Available, Finished, False)

In [14]:
df = df.withColumn("delivery_time",when(col("online_order") == "No", None)\
.otherwise(col("delivery_time"))
)

df = df.withColumn("distance_km",when(col("online_order") == "No", None)\
.otherwise(col("distance_km"))
)

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 16, Finished, Available, Finished, False)

In [15]:
display(df)

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 17, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 27ee86f0-a739-44da-9701-28678c5c3ff1)

In [16]:
df.write.format("delta") \
    .mode("append") \
    .saveAsTable("silver_orders")

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 18, Finished, Available, Finished, False)

In [17]:
spark.sql("TRUNCATE TABLE swiggy_incremental")

StatementMeta(, db6bc5bc-675b-4e7a-8d84-81be2ec264bd, 19, Finished, Available, Finished, False)

DataFrame[num_affected_rows: bigint]